In [ ]:
## Nikolay Vorontsov,
## Mushroom task, 2024
## Translate validation set into different languages.

In [1]:

from google.colab import drive


In [2]:
!pip install googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.9 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17397 sha256=8643087d124551f6f8c25be7384dfbaf3c405fd59576323683d5edcc284157d1
  Stored in directory: /root/.cache/pip/wheels/c0/59/9f/7372f0cf70160fe61b528532e1a7c8498c4becd6bcffb022de
Successfully built googletrans
  Attempting uninstall: h11
    Found existing installation: h11 0.14.0
    Uninstalling h11-0.14.0:
      Succ

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
languages = ["ar", "de", "en", "es", "fi", "fr", "hi", "it", "zh"]
datasets_path = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/"

paths =  list(datasets_path + f"mushroom.{lang}-val.v2.jsonl" for lang in languages)


In [5]:
for x in paths:
  print(x)

/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.ar-val.v2.jsonl
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.de-val.v2.jsonl
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.en-val.v2.jsonl
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.es-val.v2.jsonl
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.fi-val.v2.jsonl
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.fr-val.v2.jsonl
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.hi-val.v2.jsonl
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.it-val.v2.jsonl
/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2.jsonl


In [15]:
import json
from googletrans import Translator
import re

DELIMITER_START = "<<<"
DELIMITER_END = ">>>"

def format_text_with_spans(text, spans):
    sorted_spans = sorted(spans, key=lambda x: x[0], reverse=True)
    formatted_text = text
    for start, end in sorted_spans:
        formatted_text = formatted_text[:start] + DELIMITER_START + formatted_text[start:end] + DELIMITER_END + formatted_text[end:]
    return formatted_text

def translate_text(text, target_language):
    translator = Translator()
    try:
        translation = translator.translate(text, dest=target_language)
        return translation.text
    except Exception as e:
        print(f"Translation error: {e}")
        return text

def extract_hallucinated_words(text_with_delimiters):
    hallucinated_words = []
    for match in re.finditer(re.escape(DELIMITER_START) + "(.*?)" + re.escape(DELIMITER_END), text_with_delimiters):
        hallucinated_words.append(match.group(1))
    return hallucinated_words

def remove_delimiters(text):
    return text.replace(DELIMITER_START, "").replace(DELIMITER_END, "")

def process_jsonl(input_file, output_file, target_language='en'):
    with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
        for line in infile:
            try:
                data = json.loads(line.strip())

                lang = data.get("lang", "unknown")
                model_input = data.get("model_input", "")
                model_output_text = data.get("model_output_text", "")
                hard_labels = data.get("hard_labels", [])

                formatted_text = format_text_with_spans(model_output_text, hard_labels)
                translated_formatted_text = translate_text(formatted_text, target_language)
                translated_model_input = translate_text(model_input, target_language)

                hallucinated_words = extract_hallucinated_words(translated_formatted_text)
                translated_text_no_delimiters = remove_delimiters(translated_formatted_text)

                output_data = {
                    'lang': lang,
                    'model_input': translated_model_input,
                    'model_output_text_formatted': translated_text_no_delimiters,
                    'hallucinated_words': hallucinated_words
                }

                json.dump(output_data, outfile, ensure_ascii=False)
                outfile.write("\n")

            except json.JSONDecodeError as e:
                print(f"Error decoding JSON line: {line.strip()} - Error: {e}")
            except Exception as e:
                print(f"An unexpected error occurred: {e}")

    print(f"Processed and translated data saved to {output_file}")

In [ ]:
#for file in paths:
#  output_jsonl = f"{file}".replace(".jsonl",f"-formatted.jsonl")
#  process_jsonl(file, output_jsonl)

Processed data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2-formatted.jsonl
Processed data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2-formatted.jsonl
Processed data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2-formatted.jsonl
Processed data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2-formatted.jsonl
Processed data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2-formatted.jsonl
Processed data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2-formatted.jsonl
Processed data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2-formatted.jsonl
Processed data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/val/mushroom.zh-val.v2-formatted.jsonl
Processe

In [16]:
output_path = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/VALIDATION SET_TRANSLATE/"
for lang in ["fr", "de", "it"]:
  if lang != "en":
    source = datasets_path + f"mushroom.{lang}-val.v2.jsonl"
    target = output_path + f"mushroom.{lang}-val.v2-translated_into_en.jsonl"
    process_jsonl(source, target, target_language='en')


Processed and translated data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/VALIDATION SET_TRANSLATE/mushroom.fr-val.v2-translated_into_en.jsonl
Processed and translated data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/VALIDATION SET_TRANSLATE/mushroom.de-val.v2-translated_into_en.jsonl
Processed and translated data saved to /content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/VALIDATION SET_TRANSLATE/mushroom.it-val.v2-translated_into_en.jsonl
